In [3]:
import s1_orbits
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np

# 我们需要一个Sentinel-1场景ID来获取对应的轨道文件
# 示例场景ID格式：S1A_IW_SLC__1SDV_20230727T075102_20230727T075131_049606_05F70A_AE0A
scene_id = "S1A_IW_SLC__1SDV_20230727T075102_20230727T075131_049606_05F70A_AE0A"

# 下载对应的轨道文件
orbit_file = s1_orbits.fetch_for_scene(scene_id)

# 读取并打印文件内容
with open(orbit_file, 'r') as f:
    print("文件头部内容：")
    print(f.read(20000))  # 读取前1000个字符

文件头部内容：
<?xml version="1.0" encoding="UTF-8" standalone="no" ?>
<Earth_Explorer_File>

  <Earth_Explorer_Header>
    <Fixed_Header>
      <File_Name>S1A_OPER_AUX_POEORB_OPOD_20230816T080815_V20230726T225942_20230728T005942</File_Name>
      <File_Description>Precise Orbit Ephemerides (POE) Orbit File</File_Description>
      <Notes></Notes>
      <Mission>Sentinel-1A</Mission>
      <File_Class>OPER</File_Class>
      <File_Type>AUX_POEORB</File_Type>
      <Validity_Period>
        <Validity_Start>UTC=2023-07-26T22:59:42</Validity_Start>
        <Validity_Stop>UTC=2023-07-28T00:59:42</Validity_Stop>
      </Validity_Period>
      <File_Version>0001</File_Version>
      <Source>
        <System>OPOD</System>
        <Creator>OPOD</Creator>
        <Creator_Version>3.4.0</Creator_Version>
        <Creation_Date>UTC=2023-08-16T08:08:15</Creation_Date>
      </Source>
    </Fixed_Header>
    <Variable_Header>
      <Ref_Frame>EARTH_FIXED</Ref_Frame>
      <Time_Reference>UTC</Time_Referen

In [4]:
import xml.etree.ElementTree as ET
from datetime import datetime
import numpy as np
from scipy.interpolate import interp1d

def parse_utc_time(utc_str):
    # 解析UTC时间字符串，格式如：UTC=2023-07-26T22:59:42.000000
    time_str = utc_str.split('=')[1]
    return datetime.strptime(time_str, '%Y-%m-%dT%H:%M:%S.%f')

def extract_osv_data(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    osv_data = []
    for osv in root.findall('.//OSV'):
        utc_time = parse_utc_time(osv.find('UTC').text)
        
        osv_info = {
            'utc_time': utc_time,
            'orbit_number': int(osv.find('Absolute_Orbit').text),
            'position': np.array([
                float(osv.find('X').text),
                float(osv.find('Y').text),
                float(osv.find('Z').text)
            ]),
            'velocity': np.array([
                float(osv.find('VX').text),
                float(osv.find('VY').text),
                float(osv.find('VZ').text)
            ]),
            'quality': osv.find('Quality').text
        }
        osv_data.append(osv_info)
    
    return osv_data

def interpolate_position(osv_data, target_time):
    # 提取时间和位置数据
    times = np.array([(d['utc_time'] - osv_data[0]['utc_time']).total_seconds() 
                     for d in osv_data])
    positions = np.array([d['position'] for d in osv_data])
    
    # 对每个坐标分量进行插值
    interpolators = [interp1d(times, positions[:, i], kind='cubic') 
                    for i in range(3)]
    
    # 计算目标时间相对于第一个数据点的秒数
    target_seconds = (target_time - osv_data[0]['utc_time']).total_seconds()
    
    # 插值计算目标时间的位置
    interpolated_position = np.array([interp(target_seconds) 
                                    for interp in interpolators])
    
    return interpolated_position

# 使用示例
if __name__ == '__main__':
    # POD文件路径
    pod_file = 'S1A_OPER_AUX_POEORB_OPOD_20230816T080815_V20230726T225942_20230728T005942.EOF'
    
    # 读取轨道数据
    osv_data = extract_osv_data(pod_file)
    
    # 设定目标时间（例如：2023-07-26 23:00:00 UTC）
    target_time = datetime(2023, 7, 26, 23, 0, 0)
    
    # 计算目标时间的卫星位置
    position = interpolate_position(osv_data, target_time)
    
    print(f'卫星在 {target_time} 的位置（ECEF坐标系）：')
    print(f'X: {position[0]:.3f} 米')
    print(f'Y: {position[1]:.3f} 米')
    print(f'Z: {position[2]:.3f} 米')

卫星在 2023-07-26 23:00:00 的位置（ECEF坐标系）：
X: 1662752.517 米
Y: -6754759.797 米
Z: 1291928.754 米


In [ ]:
import s1_orbits
import matplotlib.pyplot as plt
from datetime import datetime
import numpy as np

# 我们需要一个Sentinel-1场景ID来获取对应的轨道文件
# 示例场景ID格式：S1A_IW_SLC__1SDV_20230727T075102_20230727T075131_049606_05F70A_AE0A
scene_id = "S1A_IW_SLC__1SDV_20230727T075102_20230727T075131_049606_05F70A_AE0A"

# 下载对应的轨道文件
orbit_file = s1_orbits.fetch_for_scene(scene_id)

# 读取并打印文件内容
with open(orbit_file, 'r') as f:
    print("文件头部内容：")
    print(f.read(20000))  # 读取前1000个字符

文件头部内容：
<?xml version="1.0" encoding="UTF-8" standalone="no" ?>
<Earth_Explorer_File>

  <Earth_Explorer_Header>
    <Fixed_Header>
      <File_Name>S1A_OPER_AUX_POEORB_OPOD_20230816T080815_V20230726T225942_20230728T005942</File_Name>
      <File_Description>Precise Orbit Ephemerides (POE) Orbit File</File_Description>
      <Notes></Notes>
      <Mission>Sentinel-1A</Mission>
      <File_Class>OPER</File_Class>
      <File_Type>AUX_POEORB</File_Type>
      <Validity_Period>
        <Validity_Start>UTC=2023-07-26T22:59:42</Validity_Start>
        <Validity_Stop>UTC=2023-07-28T00:59:42</Validity_Stop>
      </Validity_Period>
      <File_Version>0001</File_Version>
      <Source>
        <System>OPOD</System>
        <Creator>OPOD</Creator>
        <Creator_Version>3.4.0</Creator_Version>
        <Creation_Date>UTC=2023-08-16T08:08:15</Creation_Date>
      </Source>
    </Fixed_Header>
    <Variable_Header>
      <Ref_Frame>EARTH_FIXED</Ref_Frame>
      <Time_Reference>UTC</Time_Referen